In [4]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# 1. Load dataset
df = pd.read_csv(r"C:\Users\akshat.v\Downloads\archive(4)\Crop_recommendation.csv")

# 2. Data cleaning
df.columns = df.columns.str.strip()
df["label"] = df["label"].astype(str).str.strip().str.lower()

feature_columns = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "ph",
    "rainfall"
]

# Convert features to numeric
for column in feature_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Remove missing and duplicate rows
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

# Remove invalid values
df = df[
    (df["N"] >= 0)
    & (df["P"] >= 0)
    & (df["K"] >= 0)
    & (df["temperature"].between(-10, 60))
    & (df["humidity"].between(0, 100))
    & (df["ph"].between(0, 14))
    & (df["rainfall"] >= 0)
]

# Save cleaned dataset
df.to_csv("Crop_recommendation_cleaned.csv", index=False)

print("Cleaned dataset shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

# 3. Separate features and target
X = df[feature_columns]
y = df["label"]

# 4. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 5. Feature scaling
# Scaling is especially important for Logistic Regression, KNN and SVM
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Create models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=5000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "SVM": SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    )
}

# Models that require scaled data
scaled_models = [
    "Logistic Regression",
    "KNN",
    "SVM"
]

# 7. Train and evaluate
results = []

for model_name, model in models.items():

    if model_name in scaled_models:
        model.fit(X_train_scaled, y_train)
        predictions = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy
    })

    print("\n" + "=" * 55)
    print(model_name)
    print("=" * 55)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, predictions))

# 8. Model comparison
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

print("\nModel Comparison:")
print(results_df)

results_df.to_csv("model_comparison.csv", index=False)

Cleaned dataset shape: (2200, 8)
Missing values: 0
Duplicate rows: 0

Logistic Regression
Accuracy: 97.27%

Classification Report:
              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       0.95      1.00      0.98        20
    chickpea       1.00      1.00      1.00        20
     coconut       0.95      1.00      0.98        20
      coffee       1.00      1.00      1.00        20
      cotton       0.95      1.00      0.98        20
      grapes       1.00      1.00      1.00        20
        jute       0.83      1.00      0.91        20
 kidneybeans       1.00      1.00      1.00        20
      lentil       0.94      0.85      0.89        20
       maize       1.00      0.95      0.97        20
       mango       0.95      1.00      0.98        20
   mothbeans       0.90      0.90      0.90        20
    mungbean       1.00      1.00      1.00        20
   m

In [2]:
"""
train_model.py
================
Loads the Crop_recommendation.csv dataset, trains a RandomForest classifier,
performs hyperparameter tuning (GridSearchCV/RandomizedSearchCV) to push
accuracy as high as possible, evaluates the tuned model, and persists all
artifacts needed by the Flask app (model, scaler, label encoder, meta).
"""

import json
import time
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

DATA_PATH = "C:\\Users\\akshat.v\\Downloads\\archive(4)\\Crop_recommendation.csv"
MODEL_DIR = "models"
FEATURES = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
TARGET = "label"
RANDOM_STATE = 42


def load_data(path: str = DATA_PATH) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    return df


def build_pipeline_data(df: pd.DataFrame):
    X = df[FEATURES].values
    y_raw = df[TARGET].values

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y_raw)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled, y_train, y_test, scaler, label_encoder


def tune_random_forest(X_train, y_train):
    """Randomized search over RandomForest hyperparameters."""
    param_dist = {
        "n_estimators": [100, 150, 200, 300],
        "max_depth": [None, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"],
    }
    base_model = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=param_dist,
        n_iter=20,
        cv=cv,
        scoring="accuracy",
        n_jobs=1,
        random_state=RANDOM_STATE,
        verbose=1,
    )
    t0 = time.time()
    search.fit(X_train, y_train)
    elapsed = time.time() - t0
    print(f"RandomForest randomized search finished in {elapsed:.1f}s")
    print("Best RF params:", search.best_params_)
    print("Best RF CV accuracy: {:.4f}".format(search.best_score_))
    return search.best_estimator_, search.best_params_, search.best_score_


def tune_svc(X_train, y_train):
    """Grid search over SVC hyperparameters (probability=True for ranking)."""
    param_grid = {
        "C": [1, 10, 50],
        "gamma": ["scale", 0.1],
        "kernel": ["rbf"],
    }
    base_model = SVC(probability=True, random_state=RANDOM_STATE)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    grid = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=cv,
        scoring="accuracy",
        n_jobs=1,
        verbose=1,
    )
    t0 = time.time()
    grid.fit(X_train, y_train)
    elapsed = time.time() - t0
    print(f"SVC grid search finished in {elapsed:.1f}s")
    print("Best SVC params:", grid.best_params_)
    print("Best SVC CV accuracy: {:.4f}".format(grid.best_score_))
    return grid.best_estimator_, grid.best_params_, grid.best_score_


def tune_knn(X_train, y_train):
    param_grid = {
        "n_neighbors": [3, 5, 7, 9, 11],
        "weights": ["uniform", "distance"],
        "p": [1, 2],
    }
    base_model = KNeighborsClassifier()
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    grid = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=cv,
        scoring="accuracy",
        n_jobs=1,
        verbose=1,
    )
    grid.fit(X_train, y_train)
    print("Best KNN params:", grid.best_params_)
    print("Best KNN CV accuracy: {:.4f}".format(grid.best_score_))
    return grid.best_estimator_, grid.best_params_, grid.best_score_


def evaluate(model, X_test, y_test, label_encoder):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"\nTest accuracy: {acc:.4f}")
    print(
        classification_report(
            y_test, y_pred, target_names=label_encoder.classes_, zero_division=0
        )
    )
    cm = confusion_matrix(y_test, y_pred)
    return acc, cm


def main():
    df = load_data()
    print(f"Loaded {len(df)} rows, {df[TARGET].nunique()} classes")

    X_train, X_test, y_train, y_test, scaler, label_encoder = build_pipeline_data(df)

    candidates = {}

    rf_model, rf_params, rf_cv_score = tune_random_forest(X_train, y_train)
    candidates["random_forest"] = (rf_model, rf_params, rf_cv_score)

    svc_model, svc_params, svc_cv_score = tune_svc(X_train, y_train)
    candidates["svc"] = (svc_model, svc_params, svc_cv_score)

    knn_model, knn_params, knn_cv_score = tune_knn(X_train, y_train)
    candidates["knn"] = (knn_model, knn_params, knn_cv_score)

    results = {}
    best_name, best_model, best_acc = None, None, -1
    for name, (model, params, cv_score) in candidates.items():
        test_acc, cm = evaluate(model, X_test, y_test, label_encoder)
        results[name] = {
            "best_params": params,
            "cv_accuracy": cv_score,
            "test_accuracy": test_acc,
        }
        if test_acc > best_acc:
            best_acc = test_acc
            best_name = name
            best_model = model

    print(f"\n=== Best model: {best_name} (test accuracy {best_acc:.4f}) ===")

    joblib.dump(best_model, f"{MODEL_DIR}/crop_model.pkl")
    joblib.dump(scaler, f"{MODEL_DIR}/scaler.pkl")
    joblib.dump(label_encoder, f"{MODEL_DIR}/label_encoder.pkl")

    meta = {
        "features": FEATURES,
        "classes": label_encoder.classes_.tolist(),
        "best_model_name": best_name,
        "best_model_params": results[best_name]["best_params"],
        "test_accuracy": best_acc,
        "all_results": results,
    }
    with open(f"{MODEL_DIR}/model_meta.json", "w") as f:
        json.dump(meta, f, indent=2, default=str)

    print(f"Artifacts saved to {MODEL_DIR}/")


if __name__ == "__main__":
    main()

Loaded 2200 rows, 22 classes
Fitting 5 folds for each of 20 candidates, totalling 100 fits
RandomForest randomized search finished in 59.0s
Best RF params: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None}
Best RF CV accuracy: 0.9955
Fitting 5 folds for each of 6 candidates, totalling 30 fits
SVC grid search finished in 7.2s
Best SVC params: {'C': 50, 'gamma': 'scale', 'kernel': 'rbf'}
Best SVC CV accuracy: 0.9830
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best KNN params: {'n_neighbors': 5, 'p': 1, 'weights': 'distance'}
Best KNN CV accuracy: 0.9750

Test accuracy: 0.9955
              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       1.00      0.95      0.97        20
    chickpea       1.00      1.00      1.00        20
     coconut       1.00      1.00      1.00        20
      coffee  

FileNotFoundError: [Errno 2] No such file or directory: 'models/crop_model.pkl'